# Оценка функции плотности вероятности по выборочным данным с помощью Python

## Теоретическая часть

Предположим, у вас есть выборка данных, возможно даже большая, и вы хотите сделать некоторые выводы на основе её функции плотности вероятности (probability density function, PDF). Если предположить, что данные распределены нормально, базовой операцией будет оценка среднего значения и стандартного отклонения, так как для **подгонки нормального распределения** требуются только эти два параметра.

Однако иногда результаты подгонки могут быть неудовлетворительными. Это может быть связано с тем, что ваша выборка не выглядит точно колоколообразной, и вам интересно, что бы произошло, если бы ваша симуляция учла этот факт.

Например, возьмем доходности акций - мы знаем, что они не распределены нормально, кроме того, существует проблема "тяжелых хвостов" (fat tails). Было бы интересно оценить функцию плотности вероятности по выборочным данным, не делая предположений о её форме.

### Ядерная оценка плотности (Kernel Density Estimation, KDE)

Одним из методов для решения этой задачи является **ядерная оценка плотности** (kernel density estimation, KDE). Идея заключается в том, чтобы разместить ядро (небольшую функцию, часто гауссову) в каждой точке данных, а затем сложить все эти ядра, чтобы получить оценку плотности.

Формально, если у нас есть выборка $X_1, X_2, ..., X_n$, то оценка плотности $\hat{f}_h(x)$ в точке $x$ определяется как:

$$\hat{f}_h(x) = \frac{1}{n} \sum_{i=1}^{n} K_h(x - X_i) = \frac{1}{nh} \sum_{i=1}^{n} K\left(\frac{x - X_i}{h}\right)$$

где:
- $K$ - ядро (неотрицательная функция, интегрируемая в единицу)
- $h > 0$ - параметр сглаживания (bandwidth)

### Выбор ядра

Наиболее часто используемые ядра:

1. **Гауссово ядро**: $K(u) = \frac{1}{\sqrt{2\pi}} e^{-\frac{1}{2}u^2}$

2. **Епанечникова ядро**: $K(u) = \frac{3}{4}(1 - u^2)$ для $|u| \leq 1$

3. **Треугольное ядро**: $K(u) = (1 - |u|)$ для $|u| \leq 1$

### Выбор параметра сглаживания (bandwidth)

Параметр $h$ контролирует компромисс между смещением и дисперсией оценки:
- Слишком маленькое $h$: оценка будет шумной (высокая дисперсия)
- Слишком большое $h$: оценка будет слишком сглаженной (высокое смещение)

Правило Сильвермана (Silverman's rule of thumb) для гауссова ядра:

$$h = \left(\frac{4\hat{\sigma}^5}{3n}\right)^{\frac{1}{5}} \approx 1.06\hat{\sigma}n^{-1/5}$$

где $\hat{\sigma}$ - стандартное отклонение выборки.

## Практическая реализация

Теперь перейдем к практической реализации оценки плотности с использованием Python.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV
import warnings

# Игнорируем предупреждения для чистоты вывода
warnings.filterwarnings('ignore')

# Генерация синтетических данных
np.random.seed(42)
n_samples = 1000

# Создаем смесь двух нормальных распределений
data1 = np.random.normal(0, 1, int(0.7 * n_samples))
data2 = np.random.normal(5, 1.5, int(0.3 * n_samples))
data = np.concatenate([data1, data2])

# Вычисляем гистограмму для визуализации
hist, bin_edges = np.histogram(data, bins=30, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Параметры для построения графиков
x_plot = np.linspace(data.min() - 1, data.max() + 1, 1000)

# 1. Нормальное распределение, подобранное по данным
mu, sigma = np.mean(data), np.std(data)
normal_pdf = stats.norm(mu, sigma).pdf(x_plot)

# 2. Ядерная оценка плотности с помощью scipy
kde_scipy = stats.gaussian_kde(data)
kde_scipy_vals = kde_scipy(x_plot)

# 3. Ядерная оценка плотности с помощью sklearn
# Используем более разумный диапазон для bandwidth
silverman_bandwidth = 1.06 * sigma * len(data)**(-0.2)
bandwidths = np.linspace(silverman_bandwidth * 0.3, silverman_bandwidth * 3, 20)

# Подбор оптимального параметра bandwidth
grid = GridSearchCV(KernelDensity(kernel='gaussian'), 
                    {'bandwidth': bandwidths}, 
                    cv=5)
grid.fit(data[:, None])
best_bandwidth = grid.best_params_['bandwidth']

# Создание KDE с оптимальным bandwidth
kde_sklearn = KernelDensity(kernel='gaussian', bandwidth=best_bandwidth)
kde_sklearn.fit(data[:, None])
kde_sklearn_vals = np.exp(kde_sklearn.score_samples(x_plot[:, None]))

# Визуализация результатов
plt.figure(figsize=(12, 8))

# Гистограмма данных
plt.bar(bin_centers, hist, width=bin_edges[1]-bin_edges[0], 
        alpha=0.3, color='gray', label='Гистограмма')

# Нормальное распределение
plt.plot(x_plot, normal_pdf, 'r-', linewidth=2, 
         label=f'Нормальное распределение (μ={mu:.2f}, σ={sigma:.2f})')

# KDE от scipy
plt.plot(x_plot, kde_scipy_vals, 'b-', linewidth=2, 
         label='KDE (scipy)')

# KDE от sklearn
plt.plot(x_plot, kde_sklearn_vals, 'g--', linewidth=2, 
         label=f'KDE (sklearn, bandwidth={best_bandwidth:.2f})')

plt.xlabel('Значение')
plt.ylabel('Плотность вероятности')
plt.title('Сравнение методов оценки плотности вероятности')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Вывод статистической информации
print(f"Количество образцов: {len(data)}")
print(f"Среднее значение: {mu:.4f}")
print(f"Стандартное отклонение: {sigma:.4f}")
print(f"Оптимальный bandwidth (sklearn): {best_bandwidth:.4f}")
print(f"Bandwidth по правилу Сильвермана: {silverman_bandwidth:.4f}")
print(f"Bandwidth от scipy: {kde_scipy.factor * sigma:.4f}")

### Формулы и объяснения

#### Формула ядерной оценки плотности:

$$\hat{f}_h(x) = \frac{1}{n} \sum_{i=1}^{n} K_h(x - X_i) = \frac{1}{nh} \sum_{i=1}^{n} K\left(\frac{x - X_i}{h}\right)$$

**Объяснение**: Эта формула описывает процесс оценки плотности в точке $x$. Каждая точка данных $X_i$ вносит вклад в оценку плотности через ядерную функцию $K$, масштабированную параметром $h$. Суммирование по всем точкам данных и нормализация на $n$ дает оценку плотности.

#### Гауссово ядро:

$$K(u) = \frac{1}{\sqrt{2\pi}} e^{-\frac{1}{2}u^2}$$

**Объяснение**: Гауссово (нормальное) ядро является наиболее популярным выбором благодаря своим гладким свойствам и бесконечной поддержке. Оно придает каждой точке данных "вес", который убывает экспоненциально с расстоянием от центра ядра.

#### Правило Сильвермана для выбора bandwidth:

$$h = \left(\frac{4\hat{\sigma}^5}{3n}\right)^{\frac{1}{5}} \approx 1.06\hat{\sigma}n^{-1/5}$$

**Объяснение**: Это эмпирическое правило для выбора оптимального параметра сглаживания $h$ для гауссова ядра. Оно основано на предположении о нормальности данных и минимизирует среднеквадратичную ошибку оценки плотности.

In [ ]:
# Дополнительный анализ: сравнение разных ядер

kernels = ['gaussian', 'tophat', 'epanechnikov', 'exponential', 'linear', 'cosine']

plt.figure(figsize=(14, 10))

for i, kernel in enumerate(kernels, 1):
    plt.subplot(2, 3, i)
    
    # Используем bandwidth по правилу Сильвермана как отправную точку
    base_bandwidth = silverman_bandwidth
    
    # Для разных ядер могут потребоваться разные диапазоны bandwidth
    if kernel in ['tophat', 'epanechnikov', 'linear', 'cosine']:
        bandwidth_range = np.linspace(base_bandwidth * 0.5, base_bandwidth * 2, 15)
    else:
        bandwidth_range = np.linspace(base_bandwidth * 0.3, base_bandwidth * 3, 15)
    
    # Подбор оптимального bandwidth для каждого ядра
    grid = GridSearchCV(KernelDensity(kernel=kernel), 
                        {'bandwidth': bandwidth_range}, 
                        cv=3)  # Уменьшаем CV для скорости
    grid.fit(data[:, None])
    best_bw = grid.best_params_['bandwidth']
    
    # Создание KDE
    kde = KernelDensity(kernel=kernel, bandwidth=best_bw)
    kde.fit(data[:, None])
    log_dens = kde.score_samples(x_plot[:, None])
    
    # Построение гистограммы
    plt.hist(data, bins=30, density=True, alpha=0.3, color='gray')
    
    # Построение KDE
    plt.plot(x_plot, np.exp(log_dens), 'b-', linewidth=2)
    
    plt.title(f'Ядро: {kernel}\nBandwidth: {best_bw:.3f}')
    plt.xlabel('Значение')
    plt.ylabel('Плотность')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Анализ влияния bandwidth на оценку плотности
bandwidth_values = [silverman_bandwidth * 0.3, silverman_bandwidth * 0.7, 
                   silverman_bandwidth, silverman_bandwidth * 1.5, 
                   silverman_bandwidth * 2.5]

plt.figure(figsize=(12, 8))

for bw in bandwidth_values:
    kde = KernelDensity(kernel='gaussian', bandwidth=bw)
    kde.fit(data[:, None])
    log_dens = kde.score_samples(x_plot[:, None])
    
    plt.plot(x_plot, np.exp(log_dens), linewidth=2, 
             label=f'Bandwidth = {bw:.3f}')

plt.hist(data, bins=30, density=True, alpha=0.3, color='gray', label='Гистограмма')
plt.title('Влияние параметра bandwidth на оценку плотности')
plt.xlabel('Значение')
plt.ylabel('Плотность')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Включим предупреждения обратно
warnings.filterwarnings('default')

### Дополнительные формулы

#### Епанечникова ядро:

$$K(u) = \frac{3}{4}(1 - u^2) \quad \text{для} \quad |u| \leq 1$$

**Объяснение**: Епанечникова ядро имеет конечную поддержку и является оптимальным по критерию минимизации среднеквадратичной ошибки среди всех ядер с конечной поддержкой.

#### Треугольное ядро:

$$K(u) = (1 - |u|) \quad \text{для} \quad |u| \leq 1$$

**Объяснение**: Треугольное ядро имеет линейно убывающий вес от центра к краям поддержки, что может быть полезно в некоторых приложениях.

#### Среднеквадратичная ошибка (MSE) для оценки плотности:

$$MSE(\hat{f}_h(x)) = E[(\hat{f}_h(x) - f(x))^2] = Var(\hat{f}_h(x)) + [Bias(\hat{f}_h(x))]^2$$

**Объяснение**: Выбор параметра bandwidth представляет собой компромисс между смещением (bias) и дисперсией (variance) оценки. Слишком маленькое значение bandwidth увеличивает дисперсию, слишком большое - увеличивает смещение.

## Заключение

Ядерная оценка плотности является мощным непараметрическим методом для оценки функции плотности вероятности по выборочным данным. Ключевыми аспектами являются:

1. **Выбор ядра**: влияет на гладкость оценки
2. **Выбор bandwidth**: контролирует компромисс между смещением и дисперсией
3. **Вычислительная эффективность**: для больших datasets могут потребоваться оптимизированные алгоритмы

Метод особенно полезен, когда форма распределения неизвестна или данные имеют сложную структуру, которую нельзя описать простыми параметрическими распределениями.

### Преимущества KDE:
- Не требует предположений о форме распределения
- Может улавливать сложные модальности данных
- Обеспечивает гладкую оценку плотности

### Ограничения KDE:
- Вычислительно интенсивен для больших datasets
- Выбор bandwidth критически важен для качества оценки
- Может быть чувствителен к выбросам